# Modal single-GPU overfitting pilot v2

This experiment asks whether the complete FLUX transformer can exactly
reconstruct fixed ARC targets from conditioning prefixes that occurred
verbatim during training. NVARC and the FLUX VAE remain frozen.

The primary panel contains one exact `train` augmentation per task and is
generated at step zero and after every epoch. `train_original` and
`train_unseen` are secondary same-task transfer panels, not held-out task
evaluation.

Protocol:

- 8 tasks, 64 training augmentations per task, and 1,664 target exposures
  per epoch;
- effective and physical FLUX batch size 8;
- at most 12 manually triggered epochs;
- seen generation after every epoch;
- canonical and unseen generation after epochs 4, 8, and 12;
- explicit locally staged, committed, reloaded, and verified checkpoint
  finalization before optional automatic shutdown.

Prerequisites:

- one Modal H200;
- the writable `diffusion-arc-models` Volume mounted at
  `/mnt/diffusion-arc-models`;
- the full ARC-AGI-2 Volume setup notebook completed;
- a `GITHUB_TOKEN` Modal Secret;
- `DIFFUSION_ARC_REVISION` set to the exact pushed commit containing this
  notebook's source APIs (default: `main`).

Run top-to-bottom. Re-run the epoch cell once for each additional paid
epoch, inspect the seen-panel metrics, then explicitly run the checkpoint
finalization cell before shutdown.


In [2]:
!nvidia-smi


Tue Aug 25 15:06:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200                    On  |   00000000:A5:00.0 Off |                    0 |
| N/A   29C    P0             73W /  700W |       4MiB / 143771MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
%uv pip uninstall torchvision


Using Python 3.12.6 environment at: /usr/local
Uninstalled 1 package in 194ms
 - torchvision==0.23.0+cu129
Note: you may need to restart the kernel to use updated packages.


## 1. Immutable experiment configuration and durable report

The report is written after every meaningful state transition. Small
report writes do not establish checkpoint durability; the explicit
finalization cell later commits and reloads the mounted Volume.


In [3]:
from __future__ import annotations

import base64
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import time
import uuid

SEED = 167
PROBE_SEED = 1167
GENERATION_SEEDS = (167, 168)
MAX_EPOCHS = 12
TRANSFER_EPOCHS = (4, 8, 12)
STEPS_PER_EPOCH = 208
PROBE_INTERVAL_STEPS = 52
MAX_GRAD_NORM = 1.0
INFERENCE_STEPS = 50
EXPECTED_TRAIN_SEQUENCES = 512
EXPECTED_TRAIN_TARGETS = 1664
EXPECTED_UNSEEN_SEQUENCES = 128
EXPECTED_UNSEEN_TARGETS = 416
EXPECTED_TASKS = 8
EXPECTED_OPTIMIZER_BATCH_SIZE = 8
EXPECTED_FLUX_MICROBATCH_SIZE = 8
MINIMUM_VRAM_GIB = 120.0
MINIMUM_LOCAL_CHECKPOINT_FREE_GIB = 24.0
CHECKPOINT_MAX_SHARD_SIZE = '2GB'
VOLUME_NAME = 'diffusion-arc-models'
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'

REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
SOURCE_REVISION = "58c31c495be8c7fa04cc9ee29b491ba175568e81"
if not SOURCE_REVISION:
    raise ValueError('DIFFUSION_ARC_REVISION must not be empty')

VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
MODEL_MANIFEST = VOLUME_ROOT / 'download-manifest.json'
ARC_DATA_ROOT = VOLUME_ROOT / 'data' / 'raw' / 'arc-agi-2'
ARC_DATA_MANIFEST = VOLUME_ROOT / 'data' / 'arc-agi-2-manifest.json'
RESULTS_ROOT = VOLUME_ROOT / 'results' / 'single-gpu-overfitting-pilot'

required = (
    FLUX_CHECKPOINT,
    NVARC_CHECKPOINT,
    MODEL_MANIFEST,
    ARC_DATA_ROOT,
    ARC_DATA_MANIFEST,
)
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required Volume artifacts: {missing}')

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_DIR = RESULTS_ROOT / RUN_ID
REPORT_PATH = REPORT_DIR / 'report.json'
STEPS_PATH = REPORT_DIR / 'optimizer_steps.jsonl'
IMAGES_DIR = REPORT_DIR / 'generated_images'
REPORT_DIR.mkdir(parents=True, exist_ok=False)
IMAGES_DIR.mkdir()

REPORT: dict[str, object] = {
    'schema_version': 2,
    'status': 'INITIALIZING',
    'run_id': RUN_ID,
    'question': (
        'Can the complete FLUX transformer exactly reconstruct fixed ARC '
        'targets from conditioning prefixes seen verbatim during training?'
    ),
    'protocol': {
        'version': 2,
        'seed': SEED,
        'probe_seed': PROBE_SEED,
        'generation_seeds': list(GENERATION_SEEDS),
        'maximum_epochs': MAX_EPOCHS,
        'transfer_epochs': list(TRANSFER_EPOCHS),
        'steps_per_epoch': STEPS_PER_EPOCH,
        'probe_interval_steps': PROBE_INTERVAL_STEPS,
        'max_grad_norm': MAX_GRAD_NORM,
        'inference_steps': INFERENCE_STEPS,
        'checkpoint_max_shard_size': CHECKPOINT_MAX_SHARD_SIZE,
        'automatic_shutdown': AUTO_SHUTDOWN,
        'success_criteria': {
            'primary_panel': 'seen_training_augmentation',
            'maximum_final_train_probe_loss_ratio': 0.5,
            'minimum_exact_grid_pass2': 1,
            'strong_exact_grid_pass2': 6,
            'complete_exact_grid_pass2': 8,
        },
    },
    'paths': {
        'volume_root': str(VOLUME_ROOT),
        'flux_checkpoint': str(FLUX_CHECKPOINT),
        'nvarc_checkpoint': str(NVARC_CHECKPOINT),
        'arc_data_root': str(ARC_DATA_ROOT),
        'report': str(REPORT_PATH),
        'step_log': str(STEPS_PATH),
    },
    'failures': [],
    'probes': [],
    'seen_generation': [],
    'generation': [],
    'unseen_generation': [],
    'epochs': [],
}


def save_report() -> None:
    temporary = REPORT_PATH.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(REPORT_PATH)
    if hasattr(os, 'sync'):
        os.sync()


def record_failure(stage: str, error: BaseException) -> None:
    failure = {
        'stage': stage,
        'type': type(error).__name__,
        'message': str(error),
        'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    failures = REPORT.setdefault('failures', [])
    if isinstance(failures, list):
        failures.append(failure)
    REPORT['status'] = 'FAIL'
    REPORT['failed_stage'] = stage
    save_report()


def run_command(label: str, command: list[str], *, cwd: Path | None = None, env=None) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-40:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


save_report()
print('Run ID:', RUN_ID)
print('Durable report:', REPORT_PATH)


Run ID: 20260825T150620231361Z
Durable report: /mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T150620231361Z/report.json


## 2. Fetch, install, and test the exact Diffusion-ARC source


In [4]:
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-overfit-v2-source-{RUN_ID}'
SOURCE_DIR.mkdir(parents=True)
run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
run_command('add origin', ['git', 'remote', 'add', 'origin', REPOSITORY_URL], cwd=SOURCE_DIR)

github_token = os.environ.get('GITHUB_TOKEN', '').strip()
if not github_token:
    error = RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    record_failure('source-authentication', error)
    raise error
credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
git_environment = os.environ.copy()
git_environment.pop('GITHUB_TOKEN', None)
git_environment.update({
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
})
try:
    run_command(
        'fetch source',
        ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
        cwd=SOURCE_DIR,
        env=git_environment,
    )
    run_command('checkout source', ['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=SOURCE_DIR)
finally:
    git_environment.clear()
    del git_environment
    del credential

RESOLVED_SOURCE_COMMIT = run_command(
    'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
)

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}


def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


requirements = [
    f'{name}=={version}'
    for name, version in PINNED_PACKAGES.items()
    if installed_version(name) != version
]
if requirements:
    run_command(
        'install pinned dependencies',
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', '--no-cache-dir', *requirements],
    )
run_command(
    'install Diffusion-ARC source',
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--no-input',
        '--no-deps',
        '--no-build-isolation',
        '-e',
        str(SOURCE_DIR),
    ],
)

source_path = str((SOURCE_DIR / 'src').resolve())
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import diffusion_arc

test_environment = os.environ.copy()
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_environment['PYTHONPATH'] = source_path
test_started = time.perf_counter()
test_process = subprocess.run(
    [sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests'],
    cwd=SOURCE_DIR,
    env=test_environment,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    check=False,
)
test_seconds = time.perf_counter() - test_started
print('\n'.join(test_process.stdout.splitlines()[-30:]))
if test_process.returncode != 0:
    error = RuntimeError(f'Unit tests returned {test_process.returncode}')
    record_failure('unit-tests', error)
    raise error

REPORT['source'] = {
    'repository': REPOSITORY_URL,
    'requested_revision': SOURCE_REVISION,
    'resolved_commit': RESOLVED_SOURCE_COMMIT,
    'path': str(SOURCE_DIR),
}
REPORT['packages'] = {name: installed_version(name) for name in PINNED_PACKAGES}
REPORT['unit_tests'] = {'returncode': 0, 'seconds': test_seconds}
save_report()
print('Source commit:', RESOLVED_SOURCE_COMMIT)
print(f'Unit tests: PASS in {test_seconds:.2f}s')


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
....................................................................................................................../usr/local/lib/python3.12/site-packages/torch/nn/functional.py:3006: UserWarning: Mismatch dtype between input and weight: input dtype = c10::BFloat16, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/layer_norm.cpp:346.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
..................................
----------------------------------------------------------------------
Ran 152 tests in 6.729s

OK
Source commit: 58c31c495be8c7fa04cc9ee29b491ba175568e81
Unit tests: PASS in 30.23s


In [5]:
print(SOURCE_DIR)
print(RESOLVED_SOURCE_COMMIT)

/tmp/diffusion-arc-overfit-v2-source-20260825T150620231361Z
58c31c495be8c7fa04cc9ee29b491ba175568e81


## 3. Validate CUDA, model provenance, dataset, and v2 run configuration


In [6]:
import torch

try:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable; select an H200')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('Selected CUDA device does not support BF16')
    DEVICE = torch.device('cuda:0')
    torch.cuda.set_device(DEVICE)
    properties = torch.cuda.get_device_properties(DEVICE)
    gpu_vram_gib = properties.total_memory / 2**30
    if 'H200' not in properties.name.upper() or gpu_vram_gib < MINIMUM_VRAM_GIB:
        raise RuntimeError(
            f'Requires one H200 with >= {MINIMUM_VRAM_GIB:.0f} GiB; '
            f'received {properties.name} with {gpu_vram_gib:.1f} GiB'
        )
except Exception as error:
    record_failure('cuda-contract', error)
    raise


def cuda_memory(label: str) -> dict[str, object]:
    torch.cuda.synchronize(DEVICE)
    return {
        'label': label,
        'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
        'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
        'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
    }


model_manifest = json.loads(MODEL_MANIFEST.read_text(encoding='utf-8'))
if Path(model_manifest['flux']['path']).resolve() != FLUX_CHECKPOINT.resolve():
    raise RuntimeError('FLUX checkpoint path differs from model manifest')
if Path(model_manifest['nvarc']['path']).resolve() != NVARC_CHECKPOINT.resolve():
    raise RuntimeError('NVARC checkpoint path differs from model manifest')

dataset_manifest = json.loads(ARC_DATA_MANIFEST.read_text(encoding='utf-8'))
expected_dataset_source = {
    'repository': 'https://github.com/arcprize/ARC-AGI-2',
    'commit': 'f3283f727488ad98fe575ea6a5ac981e4a188e49',
    'archive_sha256': '179681bc01d55810ee0e90d2717ef04b8d130496541c4853479a8d6fffa4b975',
}
if dataset_manifest.get('source') != expected_dataset_source:
    raise RuntimeError('ARC dataset source identity differs')
if dataset_manifest.get('counts') != {'training': 1000, 'evaluation': 120}:
    raise RuntimeError('ARC dataset task counts differ')
for relative, expected_hash in dataset_manifest['files'].items():
    path = ARC_DATA_ROOT / relative
    if not path.is_file() or sha256_file(path) != expected_hash:
        raise RuntimeError(f'ARC dataset file differs: {relative}')

TEMP_ROOT = Path('/tmp') / f'diffusion-arc-overfit-v2-data-{RUN_ID}'
TEMP_DATA_ROOT = TEMP_ROOT / 'data'
RUNTIME_SPLIT_ROOT = TEMP_DATA_ROOT / 'splits' / 'overfitting_run_1'
RUNTIME_SPLIT_ROOT.parent.mkdir(parents=True)
shutil.copytree(
    SOURCE_DIR / 'data' / 'splits' / 'overfitting_run_1',
    RUNTIME_SPLIT_ROOT,
)
(TEMP_DATA_ROOT / 'raw').mkdir()
os.symlink(ARC_DATA_ROOT, TEMP_DATA_ROOT / 'raw' / 'arc-agi-2', target_is_directory=True)

from diffusion_arc.arc_data import parse_augmentation_key
from diffusion_arc.training.single_gpu_config import load_single_gpu_training_config

source_config_path = SOURCE_DIR / 'configs' / 'training' / 'single_gpu_overfitting_pilot_v2.json'
if not source_config_path.is_file():
    raise RuntimeError(
        'The selected source revision lacks single_gpu_overfitting_pilot_v2.json; '
        'push the implementation and select that commit.'
    )
source_config = load_single_gpu_training_config(source_config_path)
RUN_CONFIG = replace(
    source_config,
    models=replace(
        source_config.models,
        flux_checkpoint=FLUX_CHECKPOINT,
        nvarc_checkpoint=NVARC_CHECKPOINT,
    ),
    data=replace(source_config.data, split_directory=RUNTIME_SPLIT_ROOT),
)


def count_view(path: Path) -> tuple[int, int]:
    values = json.loads(path.read_text(encoding='utf-8'))
    targets = sum(
        len(parse_augmentation_key(value).augmentation.pair_indices) - 1
        for value in values
    )
    return len(values), targets


train_counts = count_view(RUNTIME_SPLIT_ROOT / 'augmented_keys' / 'train.json')
unseen_counts = count_view(RUNTIME_SPLIT_ROOT / 'augmented_keys' / 'train_unseen.json')
task_ids = tuple(json.loads((RUNTIME_SPLIT_ROOT / 'train.json').read_text()))
expected_contract = {
    'train_sequences': EXPECTED_TRAIN_SEQUENCES,
    'train_targets': EXPECTED_TRAIN_TARGETS,
    'unseen_sequences': EXPECTED_UNSEEN_SEQUENCES,
    'unseen_targets': EXPECTED_UNSEEN_TARGETS,
    'tasks': EXPECTED_TASKS,
    'steps_per_epoch': STEPS_PER_EPOCH,
    'optimizer_batch_size': EXPECTED_OPTIMIZER_BATCH_SIZE,
    'flux_microbatch_size': EXPECTED_FLUX_MICROBATCH_SIZE,
}
actual_contract = {
    'train_sequences': train_counts[0],
    'train_targets': train_counts[1],
    'unseen_sequences': unseen_counts[0],
    'unseen_targets': unseen_counts[1],
    'tasks': len(task_ids),
    'steps_per_epoch': train_counts[1] // RUN_CONFIG.loop.optimizer_batch_size,
    'optimizer_batch_size': RUN_CONFIG.loop.optimizer_batch_size,
    'flux_microbatch_size': RUN_CONFIG.loop.flux_microbatch_size,
}
if actual_contract != expected_contract:
    raise RuntimeError(f'Dataset/run contract differs: {actual_contract}')
if RUN_CONFIG.loop.max_optimizer_steps != STEPS_PER_EPOCH:
    raise RuntimeError('Configured optimizer-step count differs')

REPORT['environment'] = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu_name': properties.name,
    'gpu_vram_gib': gpu_vram_gib,
    'compute_capability': list(torch.cuda.get_device_capability(DEVICE)),
    'bf16_supported': torch.cuda.is_bf16_supported(),
}
REPORT['model_manifest_sha256'] = sha256_file(MODEL_MANIFEST)
REPORT['dataset'] = {
    'manifest_sha256': sha256_file(ARC_DATA_MANIFEST),
    'source': dataset_manifest['source'],
    'counts': dataset_manifest['counts'],
    'run_contract': actual_contract,
    'task_ids': list(task_ids),
    'split_file_sha256': {
        str(path.relative_to(RUNTIME_SPLIT_ROOT)): sha256_file(path)
        for path in sorted(RUNTIME_SPLIT_ROOT.rglob('*.json'))
    },
}
REPORT['resolved_training_config'] = json.loads(
    json.dumps(asdict(RUN_CONFIG), default=str)
)
save_report()
print(actual_contract)
print(f'GPU: {properties.name} | {gpu_vram_gib:.1f} GiB')


{'train_sequences': 512, 'train_targets': 1664, 'unseen_sequences': 128, 'unseen_targets': 416, 'tasks': 8, 'steps_per_epoch': 208, 'optimizer_batch_size': 8, 'flux_microbatch_size': 8}
GPU: NVIDIA H200 | 139.8 GiB


## 4. Load frozen NVARC/VAE and the complete trainable FLUX transformer


In [7]:
from diffusion_arc.training.single_gpu_setup import (
    build_flow_generator,
    build_flux_optimizer,
    load_single_gpu_models,
    prepare_single_gpu_device,
    seed_single_gpu_process,
)

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = prepare_single_gpu_device(RUN_CONFIG)
seed_single_gpu_process(SEED)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

load_started = time.perf_counter()
models = load_single_gpu_models(RUN_CONFIG, device=device)
optimizer = build_flux_optimizer(RUN_CONFIG, transformer=models.transformer)
flow_generator = build_flow_generator(RUN_CONFIG, device=device)
load_seconds = time.perf_counter() - load_started

if any(parameter.requires_grad for parameter in models.nvarc_runner.parameters()):
    raise RuntimeError('NVARC unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.vae_encoder.parameters()):
    raise RuntimeError('VAE encoder unexpectedly trainable')
if any(parameter.requires_grad for parameter in models.vae_decoder.parameters()):
    raise RuntimeError('VAE decoder unexpectedly trainable')
if not all(parameter.requires_grad for parameter in models.transformer.parameters()):
    raise RuntimeError('Not every FLUX parameter is trainable')

REPORT['models'] = {
    'load_seconds': load_seconds,
    'flux_trainable_parameters': sum(
        parameter.numel() for parameter in models.transformer.parameters()
        if parameter.requires_grad
    ),
    'memory_after_load': cuda_memory('after-model-load'),
}
save_report()
REPORT['models']


Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

{'load_seconds': 14.815852154999988,
 'flux_trainable_parameters': 3875544576,
 'memory_after_load': {'label': 'after-model-load',
  'allocated_gib': 21.522281169891357,
  'reserved_gib': 21.564453125,
  'peak_allocated_gib': 21.522281169891357,
  'peak_reserved_gib': 21.564453125}}

## 5. Build exact seen, canonical, and unseen panels

Each panel selects the first immutable key per task and only its final
target. The reconstructed target-free evaluation tokens must equal the
exact training-format prefix before frozen NVARC conditioning is accepted.


In [8]:
from torch.utils.data import DataLoader

from diffusion_arc.arc_data import (
    ArcEvaluationTask,
    ArcQuery,
    apply_training_augmentation,
    classify_arc_cells,
    decode_arc_grids,
    load_training_task,
    render_evaluation_prompts,
    tokenize_nvarc_evaluation_sample,
)
from diffusion_arc.inference import (
    diagnose_exact_grids,
    generate_arc_batch,
    linear_timestep_schedule,
)
from diffusion_arc.training import (
    ArcTrainingDataset,
    build_flux_microbatch,
    evaluate_fixed_flow_probe,
    prepare_flux_inputs,
    run_single_gpu_training,
)


def build_final_target_panel(view_name: str) -> dict[str, object]:
    dataset = ArcTrainingDataset(
        split_directory=RUNTIME_SPLIT_ROOT,
        view_name=view_name,
        tokenizer=models.tokenizer,
    )
    selected_keys = []
    expected_grids = []
    selected_inputs = []
    token_counts = []
    seen_task_ids = set()

    for index, key in enumerate(dataset.keys):
        if key.task_id in seen_task_ids:
            continue
        training_sample = dataset[index]
        raw_task = load_training_task(dataset.source_directory / f'{key.task_id}.json')
        augmented = apply_training_augmentation(raw_task, key.augmentation)
        if len(augmented.pairs) < 2:
            raise RuntimeError(f'Selected key has fewer than two pairs: {key}')

        query_pair = augmented.pairs[-1]
        evaluation_task = ArcEvaluationTask(
            task_id=str(key),
            demonstrations=augmented.pairs[:-1],
            queries=(ArcQuery(query_index=0, input_grid=query_pair.input_grid),),
        )
        rendered = render_evaluation_prompts(evaluation_task)[0]
        tokenized = tokenize_nvarc_evaluation_sample(
            rendered,
            models.tokenizer,
            max_tokens=dataset.max_tokens,
        )
        training_prefix = training_sample.input_ids[:training_sample.prefix_token_ends[-1]]
        if tokenized.input_ids != training_prefix:
            raise RuntimeError(f'Evaluation tokens differ from training prefix: {key}')

        flux_inputs = prepare_flux_inputs(
            training_sample,
            nvarc_runner=models.nvarc_runner,
            vae_encoder=models.vae_encoder,
            min_context_pairs_before_target=len(training_sample.target_grids),
        )
        if len(flux_inputs) != 1:
            raise RuntimeError(f'Expected one final target for {key}')
        if training_sample.target_grids[-1] != query_pair.output_grid:
            raise RuntimeError(f'Augmented target differs from training sample: {key}')

        selected_keys.append(str(key))
        expected_grids.append(query_pair.output_grid)
        selected_inputs.append(flux_inputs[0])
        token_counts.append(len(tokenized.input_ids))
        seen_task_ids.add(key.task_id)

    selected_task_ids = tuple(key.split('_', 1)[0] for key in selected_keys)
    if selected_task_ids != task_ids:
        raise RuntimeError(
            f'{view_name} task order differs: {selected_task_ids!r} versus {task_ids!r}'
        )
    if len(selected_keys) != EXPECTED_TASKS:
        raise RuntimeError(
            f'Expected {EXPECTED_TASKS} keys for {view_name}, received {len(selected_keys)}'
        )

    microbatch = build_flux_microbatch(tuple(selected_inputs))
    return {
        'view_name': view_name,
        'keys': tuple(selected_keys),
        'expected_grids': tuple(expected_grids),
        'text_inputs': microbatch.text_inputs,
        'clean_latents': microbatch.clean_latents,
        'probe_batch': microbatch,
        'token_counts': tuple(token_counts),
        'exact_training_prefix_match': (True,) * len(selected_keys),
    }


PANELS = {
    'seen_training_augmentation': build_final_target_panel('train'),
    'canonical_original': build_final_target_panel('train_original'),
    'unseen_augmentation': build_final_target_panel('train_unseen'),
}
PROBE_BATCHES = {
    name: (panel['probe_batch'],)
    for name, panel in PANELS.items()
}

seen_panel = PANELS['seen_training_augmentation']
with torch.no_grad():
    decoded = models.vae_decoder(seen_panel['clean_latents'])
roundtrip_rgb = (
    decoded.detach().to(device='cpu', dtype=torch.float32)
    .clamp(-1.0, 1.0).add(1.0).mul(127.5).round().to(dtype=torch.uint8)
)
roundtrip_predictions = decode_arc_grids(classify_arc_cells(roundtrip_rgb))
roundtrip_diagnostics = diagnose_exact_grids(
    roundtrip_predictions,
    seen_panel['expected_grids'],
)
roundtrip_exact = sum(item.exact_grid for item in roundtrip_diagnostics)
if roundtrip_exact != EXPECTED_TASKS:
    raise RuntimeError(
        f'Seen-panel clean VAE round trip is {roundtrip_exact}/{EXPECTED_TASKS}, expected exact'
    )
del decoded, roundtrip_rgb, roundtrip_predictions, roundtrip_diagnostics

REPORT['evaluation_panels'] = {
    name: {
        'view_name': panel['view_name'],
        'keys': list(panel['keys']),
        'target_free_token_counts': list(panel['token_counts']),
        'expected_shapes': [
            [len(grid), len(grid[0])] for grid in panel['expected_grids']
        ],
        'exact_training_prefix_match': list(panel['exact_training_prefix_match']),
    }
    for name, panel in PANELS.items()
}
REPORT['vae_roundtrip'] = {
    'panel': 'seen_training_augmentation',
    'exact_grids': roundtrip_exact,
    'total_grids': EXPECTED_TASKS,
}
REPORT['memory_after_panels'] = cuda_memory('after-panel-preparation')
save_report()
{
    'panels': {name: list(panel['keys']) for name, panel in PANELS.items()},
    'vae_roundtrip': REPORT['vae_roundtrip'],
}


{'panels': {'seen_training_augmentation': ['3c9b0459_ID_1290374658_4,1,2,3,0',
   '0d3d703e_ID_7458913620_3,0,4,2,1',
   '8be77c9e_ID_6921504738_1,2,3,0',
   'c9e6f938_ID_7809641352_0,1,3,2',
   '00576224_ID_6095374821_1,0,2',
   '0520fde7_ID_4960851372_2,0,1,3',
   '88a62173_ID_1984750623_2,0,1,3',
   'd631b094_ID_0487193526_2,4,0,1,3'],
  'canonical_original': ['3c9b0459_ID_0123456789_0,1,2,3,4',
   '0d3d703e_ID_0123456789_0,1,2,3,4',
   '8be77c9e_ID_0123456789_0,1,2,3',
   'c9e6f938_ID_0123456789_0,1,2,3',
   '00576224_ID_0123456789_0,1,2',
   '0520fde7_ID_0123456789_0,1,2,3',
   '88a62173_ID_0123456789_0,1,2,3',
   'd631b094_ID_0123456789_0,1,2,3,4'],
  'unseen_augmentation': ['3c9b0459_ID_5896310427_4,2,1,0,3',
   '0d3d703e_ID_4912687530_0,3,4,1,2',
   '8be77c9e_ID_9741652038_1,0,3,2',
   'c9e6f938_ID_2675430198_3,2,1,0',
   '00576224_ID_0378942156_0,2,1',
   '0520fde7_ID_2830971546_0,2,1,3',
   '88a62173_ID_6017325489_2,3,0,1',
   'd631b094_ID_7169453802_4,1,0,3,2']},
 'vae_round

## 6. Probe, generation, epoch, and checkpoint helpers

`run_next_epoch()` is stateful and refuses skipped or excess epochs. Seen
generation runs after every epoch. Transfer panels run only at their
preregistered milestones or when canonical generation is needed to
finalize a checkpoint at another step.


In [9]:
from PIL import Image
from safetensors import safe_open
from tqdm.auto import tqdm

GLOBAL_STEP = 0
COMPLETED_EPOCHS = 0
TIMESTEP_SCHEDULE = linear_timestep_schedule(INFERENCE_STEPS, device=device)
GENERATION_FIELDS = {
    'seen_training_augmentation': 'seen_generation',
    'canonical_original': 'generation',
    'unseen_augmentation': 'unseen_generation',
}


def append_step_record(record: dict[str, object]) -> None:
    with STEPS_PATH.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, sort_keys=True, allow_nan=False) + '\n')
        handle.flush()
        os.fsync(handle.fileno())


def run_fixed_probes(global_step: int) -> dict[str, float]:
    losses = {}
    for name, batches in PROBE_BATCHES.items():
        result = evaluate_fixed_flow_probe(
            models.transformer,
            batches,
            compute_dtype=models.flux_compute_dtype,
            seed=PROBE_SEED,
        )
        losses[name] = result.loss
    entry = {
        'global_step': global_step,
        'losses': losses,
        'memory': cuda_memory(f'probe-step-{global_step}'),
    }
    REPORT['probes'].append(entry)
    save_report()
    print('Fixed probes:', entry)
    return losses


def diagnostic_payload(diagnostic) -> dict[str, object]:
    return {
        'prediction_valid': diagnostic.prediction_valid,
        'invalid_reason': diagnostic.invalid_reason,
        'predicted_shape': diagnostic.predicted_shape,
        'expected_shape': diagnostic.expected_shape,
        'shape_match': diagnostic.shape_match,
        'cell_accuracy': diagnostic.cell_accuracy,
        'exact_grid': diagnostic.exact_grid,
    }


def run_generation(panel_name: str, global_step: int) -> dict[str, object]:
    if panel_name not in PANELS:
        raise ValueError(f'Unknown generation panel: {panel_name}')
    report_field = GENERATION_FIELDS[panel_name]
    existing = [
        entry for entry in REPORT[report_field]
        if entry['global_step'] == global_step
    ]
    if existing:
        raise RuntimeError(
            f'{panel_name} generation already exists at step {global_step}'
        )

    panel = PANELS[panel_name]
    samples_by_seed = []
    diagnostics_by_seed = []
    generation_started = time.perf_counter()
    for generation_seed in tqdm(
        GENERATION_SEEDS,
        desc=f'{panel_name} @ step {global_step}',
        unit='seed',
    ):
        result = generate_arc_batch(
            models.transformer,
            models.vae_decoder,
            panel['text_inputs'],
            timestep_schedule=TIMESTEP_SCHEDULE,
            compute_dtype=models.flux_compute_dtype,
            generator=torch.Generator(device=device).manual_seed(generation_seed),
        )
        diagnostics = diagnose_exact_grids(
            result.predictions,
            panel['expected_grids'],
        )
        seed_samples = []
        for index, (key, prediction, diagnostic) in enumerate(
            zip(panel['keys'], result.predictions, diagnostics, strict=True)
        ):
            task_id = key.split('_', 1)[0]
            filename = (
                f'{panel_name}-step-{global_step:04d}-{index:02d}-'
                f'{task_id}-seed-{generation_seed}.png'
            )
            image = result.rgb_images[index].permute(1, 2, 0).contiguous().numpy()
            Image.fromarray(image).save(IMAGES_DIR / filename)
            seed_samples.append({
                'key': key,
                'seed': generation_seed,
                'image': str((IMAGES_DIR / filename).relative_to(REPORT_DIR)),
                'prediction': prediction.grid,
                'diagnostic': diagnostic_payload(diagnostic),
            })
        samples_by_seed.append(seed_samples)
        diagnostics_by_seed.append(diagnostics)

    exact_pass1 = sum(item.exact_grid for item in diagnostics_by_seed[0])
    exact_pass2 = sum(
        first.exact_grid or second.exact_grid
        for first, second in zip(*diagnostics_by_seed, strict=True)
    )
    valid_pass2 = sum(
        first.prediction_valid or second.prediction_valid
        for first, second in zip(*diagnostics_by_seed, strict=True)
    )
    shape_pass2 = sum(
        first.shape_match or second.shape_match
        for first, second in zip(*diagnostics_by_seed, strict=True)
    )
    best_cell_accuracies = []
    for first, second in zip(*diagnostics_by_seed, strict=True):
        available = [
            value for value in (first.cell_accuracy, second.cell_accuracy)
            if value is not None
        ]
        best_cell_accuracies.append(max(available) if available else None)

    entry = {
        'panel': panel_name,
        'view_name': panel['view_name'],
        'global_step': global_step,
        'keys': list(panel['keys']),
        'exact_grid_pass1': exact_pass1,
        'exact_grid_pass2': exact_pass2,
        'valid_grid_pass2': valid_pass2,
        'correct_shape_pass2': shape_pass2,
        'best_cell_accuracies': best_cell_accuracies,
        'seconds': time.perf_counter() - generation_started,
        'samples_by_seed': samples_by_seed,
        'memory': cuda_memory(f'{panel_name}-generation-step-{global_step}'),
    }
    REPORT[report_field].append(entry)
    save_report()
    print({key: value for key, value in entry.items() if key != 'samples_by_seed'})
    return entry


def available_mean(values):
    present = [value for value in values if value is not None]
    return None if not present else sum(present) / len(present)


def evidence_summary() -> dict[str, object]:
    if not REPORT['probes'] or not REPORT['seen_generation']:
        raise RuntimeError('Baseline probes and seen generation are missing')
    baseline_probe = REPORT['probes'][0]['losses']['seen_training_augmentation']
    final_probe = REPORT['probes'][-1]['losses']['seen_training_augmentation']
    baseline_generation = REPORT['seen_generation'][0]
    final_generation = REPORT['seen_generation'][-1]
    baseline_cell_mean = available_mean(
        baseline_generation['best_cell_accuracies']
    )
    final_cell_mean = available_mean(final_generation['best_cell_accuracies'])
    summary = {
        'primary_panel': 'seen_training_augmentation',
        'train_probe_loss_ratio': final_probe / baseline_probe,
        'optimization_signal': final_probe / baseline_probe <= 0.5,
        'positive_overfitting_evidence': final_generation['exact_grid_pass2'] >= 1,
        'strong_overfitting_evidence': final_generation['exact_grid_pass2'] >= 6,
        'complete_overfitting': final_generation['exact_grid_pass2'] == 8,
        'baseline_conditional_cell_accuracy': baseline_cell_mean,
        'final_conditional_cell_accuracy': final_cell_mean,
        'latest_seen_exact_grid_pass2': final_generation['exact_grid_pass2'],
        'latest_seen_correct_shape_pass2': final_generation['correct_shape_pass2'],
    }
    REPORT['evidence_summary'] = summary
    save_report()
    return summary


def run_next_epoch() -> dict[str, object]:
    global GLOBAL_STEP, COMPLETED_EPOCHS
    epoch = COMPLETED_EPOCHS + 1
    if not 1 <= epoch <= MAX_EPOCHS:
        raise RuntimeError(f'No epoch remains; completed {COMPLETED_EPOCHS}/{MAX_EPOCHS}')
    if REPORT.get('saved_checkpoint') is not None:
        raise RuntimeError('Training cannot continue after checkpoint finalization')

    epoch_config = replace(RUN_CONFIG, seed=SEED + epoch - 1)
    dataset = ArcTrainingDataset(
        split_directory=RUNTIME_SPLIT_ROOT,
        view_name='train',
        tokenizer=models.tokenizer,
    )
    loader = DataLoader(
        dataset,
        batch_size=None,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator(device='cpu').manual_seed(epoch_config.seed),
    )
    epoch_records = []
    epoch_started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(device)
    progress = tqdm(
        total=STEPS_PER_EPOCH,
        desc=f'Epoch {epoch}/{MAX_EPOCHS}',
        unit='update',
    )

    def on_step(metrics) -> None:
        nonlocal epoch_records
        global GLOBAL_STEP
        GLOBAL_STEP += 1
        record = {
            'epoch': epoch,
            'epoch_step': len(epoch_records) + 1,
            'global_step': GLOBAL_STEP,
            'loss': float(metrics.loss.item()),
            'target_count': metrics.target_count,
            'microbatch_count': metrics.microbatch_count,
            'gradient_norm': metrics.gradient_norm,
            'gradient_clipped': metrics.gradient_clipped,
            'learning_rates': list(metrics.learning_rates),
            'optimizer_step_seconds': metrics.step_seconds,
            'cuda_allocated_gib': torch.cuda.memory_allocated(device) / 2**30,
            'cuda_reserved_gib': torch.cuda.memory_reserved(device) / 2**30,
            'cuda_peak_allocated_gib': torch.cuda.max_memory_allocated(device) / 2**30,
        }
        epoch_records.append(record)
        append_step_record(record)
        progress.update(1)
        progress.set_postfix(
            loss=f'{record["loss"]:.5f}',
            grad=f'{record["gradient_norm"]:.3f}',
            peak_gib=f'{record["cuda_peak_allocated_gib"]:.1f}',
            refresh=False,
        )
        if GLOBAL_STEP % PROBE_INTERVAL_STEPS == 0:
            run_fixed_probes(GLOBAL_STEP)

    try:
        run_single_gpu_training(
            loader,
            nvarc_runner=models.nvarc_runner,
            vae_encoder=models.vae_encoder,
            transformer=models.transformer,
            optimizer=optimizer,
            flux_compute_dtype=models.flux_compute_dtype,
            max_optimizer_steps=STEPS_PER_EPOCH,
            optimizer_batch_size=RUN_CONFIG.loop.optimizer_batch_size,
            flux_microbatch_size=RUN_CONFIG.loop.flux_microbatch_size,
            min_context_pairs_before_target=RUN_CONFIG.loop.min_context_pairs_before_target,
            generator=flow_generator,
            max_grad_norm=MAX_GRAD_NORM,
            on_optimizer_step_metrics=on_step,
        )
    except Exception as error:
        record_failure(f'epoch-{epoch}', error)
        raise
    finally:
        progress.close()

    if len(epoch_records) != STEPS_PER_EPOCH:
        error = RuntimeError(
            f'Epoch {epoch} completed {len(epoch_records)} steps, expected {STEPS_PER_EPOCH}'
        )
        record_failure(f'epoch-{epoch}-length', error)
        raise error

    epoch_seconds = time.perf_counter() - epoch_started
    COMPLETED_EPOCHS = epoch
    try:
        seen_generation = run_generation(
            'seen_training_augmentation', GLOBAL_STEP
        )
        transfer_generation = {}
        if epoch in TRANSFER_EPOCHS:
            transfer_generation['canonical_original'] = run_generation(
                'canonical_original', GLOBAL_STEP
            )
            transfer_generation['unseen_augmentation'] = run_generation(
                'unseen_augmentation', GLOBAL_STEP
            )
    except Exception as error:
        record_failure(f'epoch-{epoch}-generation', error)
        raise

    epoch_summary = {
        'epoch': epoch,
        'first_global_step': epoch_records[0]['global_step'],
        'last_global_step': epoch_records[-1]['global_step'],
        'seconds': epoch_seconds,
        'mean_training_loss': sum(record['loss'] for record in epoch_records) / len(epoch_records),
        'mean_gradient_norm': sum(record['gradient_norm'] for record in epoch_records) / len(epoch_records),
        'clipped_updates': sum(record['gradient_clipped'] for record in epoch_records),
        'memory': cuda_memory(f'epoch-{epoch}-complete'),
        'seen_exact_grid_pass2': seen_generation['exact_grid_pass2'],
        'seen_correct_shape_pass2': seen_generation['correct_shape_pass2'],
        'transfer_panels_generated': list(transfer_generation),
    }
    REPORT['epochs'].append(epoch_summary)
    REPORT['completed_epochs'] = COMPLETED_EPOCHS
    REPORT['completed_optimizer_steps'] = GLOBAL_STEP
    REPORT['status'] = (
        'TRAINING_LIMIT_REACHED' if epoch == MAX_EPOCHS else 'READY_FOR_NEXT_EPOCH'
    )
    REPORT['evidence_summary'] = evidence_summary()
    save_report()
    return epoch_summary


def checkpoint_manifest(directory: Path) -> dict[str, dict[str, object]]:
    return {
        str(path.relative_to(directory)): {
            'bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        }
        for path in sorted(directory.rglob('*'))
        if path.is_file()
    }


def validate_transformer_checkpoint(directory: Path) -> dict[str, dict[str, object]]:
    config_path = directory / 'config.json'
    index_path = directory / 'diffusion_pytorch_model.safetensors.index.json'
    if not config_path.is_file() or not index_path.is_file():
        raise RuntimeError(f'Checkpoint lacks config or Safetensors index: {directory}')
    index = json.loads(index_path.read_text(encoding='utf-8'))
    weight_map = index.get('weight_map')
    if not isinstance(weight_map, dict) or not weight_map:
        raise RuntimeError('Safetensors index contains no weight map')
    shard_names = set(weight_map.values())
    actual_shards = {
        path.name for path in directory.glob('*.safetensors') if path.is_file()
    }
    if actual_shards != shard_names:
        raise RuntimeError(
            f'Safetensors shard set differs: {actual_shards} versus {shard_names}'
        )
    observed_keys = set()
    for shard_name in sorted(shard_names):
        with safe_open(directory / shard_name, framework='pt', device='cpu') as handle:
            shard_keys = set(handle.keys())
        if observed_keys.intersection(shard_keys):
            raise RuntimeError(f'Duplicate tensor keys in shard {shard_name}')
        observed_keys.update(shard_keys)
    if observed_keys != set(weight_map):
        raise RuntimeError('Safetensors tensor keys differ from the index')
    manifest = checkpoint_manifest(directory)
    if any(item['bytes'] <= 0 for item in manifest.values()):
        raise RuntimeError('Checkpoint contains an empty file')
    return manifest


def ensure_canonical_generation_for_current_step() -> None:
    if not any(
        entry['global_step'] == GLOBAL_STEP for entry in REPORT['generation']
    ):
        run_generation('canonical_original', GLOBAL_STEP)


def commit_and_reload_volume(volume, *, stage: str) -> None:
    started = time.perf_counter()
    volume.commit()
    volume.reload()
    print(f'{stage}: committed and reloaded in {time.perf_counter() - started:.1f}s')


def save_current_checkpoint() -> dict[str, object]:
    if GLOBAL_STEP == 0 or COMPLETED_EPOCHS == 0:
        raise RuntimeError('Refusing to save the untrained step-0 transformer')
    if REPORT.get('saved_checkpoint') is not None:
        raise RuntimeError('This run already records a finalized checkpoint')
    ensure_canonical_generation_for_current_step()

    free_gib = shutil.disk_usage('/tmp').free / 2**30
    if free_gib < MINIMUM_LOCAL_CHECKPOINT_FREE_GIB:
        raise RuntimeError(
            f'Checkpoint staging requires at least {MINIMUM_LOCAL_CHECKPOINT_FREE_GIB:.0f} '
            f'GiB free in /tmp; found {free_gib:.1f} GiB'
        )

    checkpoint_name = f'flux-transformer-step-{GLOBAL_STEP:04d}'
    local_directory = Path('/tmp') / f'{checkpoint_name}-{RUN_ID}'
    final_directory = REPORT_DIR / checkpoint_name
    staging_directory = REPORT_DIR / f'.{checkpoint_name}.partial-{uuid.uuid4().hex}'
    for path in (local_directory, final_directory, staging_directory):
        if path.exists():
            raise FileExistsError(f'Refusing to overwrite checkpoint path: {path}')

    try:
        save_started = time.perf_counter()
        print('Saving transformer to local ephemeral storage:', local_directory)
        models.transformer.save_pretrained(
            local_directory,
            safe_serialization=True,
            max_shard_size=CHECKPOINT_MAX_SHARD_SIZE,
        )
        local_manifest = validate_transformer_checkpoint(local_directory)
        print(
            f'Local checkpoint verified: {len(local_manifest)} files in '
            f'{time.perf_counter() - save_started:.1f}s'
        )

        print('Copying verified checkpoint to Volume staging:', staging_directory)
        shutil.copytree(local_directory, staging_directory)
        staging_manifest = validate_transformer_checkpoint(staging_directory)
        if staging_manifest != local_manifest:
            raise RuntimeError('Volume staging checkpoint differs from local checkpoint')

        import modal

        volume = modal.Volume.from_name(VOLUME_NAME)
        commit_and_reload_volume(volume, stage='Checkpoint staging')
        durable_staging_manifest = validate_transformer_checkpoint(staging_directory)
        if durable_staging_manifest != local_manifest:
            raise RuntimeError('Reloaded staging checkpoint differs from local checkpoint')

        staging_directory.rename(final_directory)
        commit_and_reload_volume(volume, stage='Checkpoint final rename')
        durable_final_manifest = validate_transformer_checkpoint(final_directory)
        if durable_final_manifest != local_manifest:
            raise RuntimeError('Reloaded final checkpoint differs from local checkpoint')

        REPORT['saved_checkpoint'] = {
            'global_step': GLOBAL_STEP,
            'completed_epochs': COMPLETED_EPOCHS,
            'path': str(final_directory),
            'files': durable_final_manifest,
            'local_staging_path': str(local_directory),
            'max_shard_size': CHECKPOINT_MAX_SHARD_SIZE,
            'volume_commit_verified': True,
        }
        REPORT['status'] = 'COMPLETE'
        REPORT['completion_reason'] = 'user_finalized_checkpoint'
        REPORT['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
        REPORT['evidence_summary'] = evidence_summary()
        save_report()
        commit_and_reload_volume(volume, stage='Final report')

        durable_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
        if durable_report.get('saved_checkpoint') != REPORT['saved_checkpoint']:
            raise RuntimeError('Reloaded report differs from finalized checkpoint record')
        if durable_report.get('status') != 'COMPLETE':
            raise RuntimeError('Reloaded report is not COMPLETE')
        print('Durable checkpoint and report verified:', final_directory)
        return REPORT['saved_checkpoint']
    except Exception as error:
        record_failure('checkpoint-finalization', error)
        raise


## 7. Step-zero baselines

Run once before any optimizer update. The seen and canonical panels are
generated independently with the same fixed pass@2 seeds.


In [10]:
if GLOBAL_STEP != 0 or REPORT['probes'] or REPORT['seen_generation'] or REPORT['generation']:
    raise RuntimeError('Step-zero baseline has already been recorded')
REPORT['status'] = 'BASELINE_RUNNING'
save_report()
baseline_probe_losses = run_fixed_probes(0)
baseline_seen = run_generation('seen_training_augmentation', 0)
baseline_canonical = run_generation('canonical_original', 0)
REPORT['status'] = 'READY_FOR_NEXT_EPOCH'
REPORT['evidence_summary'] = evidence_summary()
save_report()
{
    'probe_losses': baseline_probe_losses,
    'seen': {key: value for key, value in baseline_seen.items() if key != 'samples_by_seed'},
    'canonical': {
        key: value for key, value in baseline_canonical.items()
        if key != 'samples_by_seed'
    },
}


Fixed probes: {'global_step': 0, 'losses': {'seen_training_augmentation': 0.17930014431476593, 'canonical_original': 0.1656118482351303, 'unseen_augmentation': 0.1717759072780609}, 'memory': {'label': 'probe-step-0', 'allocated_gib': 21.619651317596436, 'reserved_gib': 35.408203125, 'peak_allocated_gib': 30.37626361846924, 'peak_reserved_gib': 35.408203125}}


seen_training_augmentation @ step 0:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 0, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 1, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'seconds': 17.44556255400002, 'memory': {'label': 'seen_training_augmentation-generation-step-0', 'allocated_gib': 21.619651317596436, 'reserved_gib': 35.408203125, 'peak_allocated_gib': 30.37626361846924, 'peak_reserved_gib': 35.408203125}}


canonical_original @ step 0:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'canonical_original', 'view_name': 'train_original', 'global_step': 0, 'keys': ['3c9b0459_ID_0123456789_0,1,2,3,4', '0d3d703e_ID_0123456789_0,1,2,3,4', '8be77c9e_ID_0123456789_0,1,2,3', 'c9e6f938_ID_0123456789_0,1,2,3', '00576224_ID_0123456789_0,1,2', '0520fde7_ID_0123456789_0,1,2,3', '88a62173_ID_0123456789_0,1,2,3', 'd631b094_ID_0123456789_0,1,2,3,4'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 1, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'seconds': 17.482230638999965, 'memory': {'label': 'canonical_original-generation-step-0', 'allocated_gib': 21.619651317596436, 'reserved_gib': 35.408203125, 'peak_allocated_gib': 30.37626361846924, 'peak_reserved_gib': 35.408203125}}


{'probe_losses': {'seen_training_augmentation': 0.17930014431476593,
  'canonical_original': 0.1656118482351303,
  'unseen_augmentation': 0.1717759072780609},
 'seen': {'panel': 'seen_training_augmentation',
  'view_name': 'train',
  'global_step': 0,
  'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0',
   '0d3d703e_ID_7458913620_3,0,4,2,1',
   '8be77c9e_ID_6921504738_1,2,3,0',
   'c9e6f938_ID_7809641352_0,1,3,2',
   '00576224_ID_6095374821_1,0,2',
   '0520fde7_ID_4960851372_2,0,1,3',
   '88a62173_ID_1984750623_2,0,1,3',
   'd631b094_ID_0487193526_2,4,0,1,3'],
  'exact_grid_pass1': 0,
  'exact_grid_pass2': 0,
  'valid_grid_pass2': 1,
  'correct_shape_pass2': 0,
  'best_cell_accuracies': [None, None, None, None, None, None, None, None],
  'seconds': 17.44556255400002,
  'memory': {'label': 'seen_training_augmentation-generation-step-0',
   'allocated_gib': 21.619651317596436,
   'reserved_gib': 35.408203125,
   'peak_allocated_gib': 30.37626361846924,
   'peak_reserved_gib': 35.408203125}},
 

## 8. Run exactly one next epoch

Re-run this cell manually for each additional epoch. It refuses to exceed
12 epochs or to continue after checkpoint finalization. Inspect
`seen_exact_grid_pass2` before deciding to pay for another epoch.


In [11]:
epoch_result = run_next_epoch()
epoch_result


Epoch 1/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 52, 'losses': {'seen_training_augmentation': 0.029536094516515732, 'canonical_original': 0.030443765223026276, 'unseen_augmentation': 0.029939012601971626}, 'memory': {'label': 'probe-step-52', 'allocated_gib': 64.99836826324463, 'reserved_gib': 109.361328125, 'peak_allocated_gib': 95.76475715637207, 'peak_reserved_gib': 109.361328125}}
Fixed probes: {'global_step': 104, 'losses': {'seen_training_augmentation': 0.023215176537632942, 'canonical_original': 0.02406301163136959, 'unseen_augmentation': 0.023780763149261475}, 'memory': {'label': 'probe-step-104', 'allocated_gib': 64.9986047744751, 'reserved_gib': 109.361328125, 'peak_allocated_gib': 95.76671028137207, 'peak_reserved_gib': 109.361328125}}
Fixed probes: {'global_step': 156, 'losses': {'seen_training_augmentation': 0.02827470190823078, 'canonical_original': 0.028680766001343727, 'unseen_augmentation': 0.027896471321582794}, 'memory': {'label': 'probe-step-156', 'allocated_gib': 65.0000786781311, 'r

seen_training_augmentation @ step 208:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 208, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 6, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'seconds': 16.55716427499999, 'memory': {'label': 'seen_training_augmentation-generation-step-208', 'allocated_gib': 64.96829175949097, 'reserved_gib': 119.490234375, 'peak_allocated_gib': 95.76725196838379, 'peak_reserved_gib': 119.490234375}}


{'epoch': 1,
 'first_global_step': 1,
 'last_global_step': 208,
 'seconds': 160.40412455399996,
 'mean_training_loss': 0.038504024159360245,
 'mean_gradient_norm': 1.8905664633672972,
 'clipped_updates': 117,
 'memory': {'label': 'epoch-1-complete',
  'allocated_gib': 64.96829175949097,
  'reserved_gib': 119.490234375,
  'peak_allocated_gib': 95.76725196838379,
  'peak_reserved_gib': 119.490234375},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 0,
 'transfer_panels_generated': []}

## 9. Current evidence summary

This cell does not modify model weights. Run it whenever you want a compact
decision view before continuing or finalizing.


In [12]:
summary = evidence_summary()
print('Status:', REPORT['status'])
print('Completed epochs:', COMPLETED_EPOCHS)
print('Completed optimizer steps:', GLOBAL_STEP)
print('Latest seen exact-grid pass@2:', summary['latest_seen_exact_grid_pass2'], '/ 8')
print('Latest seen correct-shape pass@2:', summary['latest_seen_correct_shape_pass2'], '/ 8')
print('Report:', REPORT_PATH)
summary


Status: READY_FOR_NEXT_EPOCH
Completed epochs: 1
Completed optimizer steps: 208
Latest seen exact-grid pass@2: 0 / 8
Latest seen correct-shape pass@2: 0 / 8
Report: /mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T150620231361Z/report.json


{'primary_panel': 'seen_training_augmentation',
 'train_probe_loss_ratio': 0.12143132851515658,
 'optimization_signal': True,
 'positive_overfitting_evidence': False,
 'strong_overfitting_evidence': False,
 'complete_overfitting': False,
 'baseline_conditional_cell_accuracy': None,
 'final_conditional_cell_accuracy': None,
 'latest_seen_exact_grid_pass2': 0,
 'latest_seen_correct_shape_pass2': 0}

In [13]:
epoch_result = run_next_epoch()
epoch_result

Epoch 2/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 260, 'losses': {'seen_training_augmentation': 0.02673155441880226, 'canonical_original': 0.02684946358203888, 'unseen_augmentation': 0.02711998112499714}, 'memory': {'label': 'probe-step-260', 'allocated_gib': 65.00266981124878, 'reserved_gib': 119.490234375, 'peak_allocated_gib': 95.76573371887207, 'peak_reserved_gib': 119.490234375}}
Fixed probes: {'global_step': 312, 'losses': {'seen_training_augmentation': 0.01972523331642151, 'canonical_original': 0.02016078121960163, 'unseen_augmentation': 0.01997307315468788}, 'memory': {'label': 'probe-step-312', 'allocated_gib': 64.99871969223022, 'reserved_gib': 119.490234375, 'peak_allocated_gib': 95.76671409606934, 'peak_reserved_gib': 119.490234375}}
Fixed probes: {'global_step': 364, 'losses': {'seen_training_augmentation': 0.01689479500055313, 'canonical_original': 0.01815471239387989, 'unseen_augmentation': 0.017093783244490623}, 'memory': {'label': 'probe-step-364', 'allocated_gib': 64.99982166290283, 'res

seen_training_augmentation @ step 416:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 416, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'seconds': 16.57047696699999, 'memory': {'label': 'seen_training_augmentation-generation-step-416', 'allocated_gib': 64.96829175949097, 'reserved_gib': 119.490234375, 'peak_allocated_gib': 95.76671409606934, 'peak_reserved_gib': 119.490234375}}


{'epoch': 2,
 'first_global_step': 209,
 'last_global_step': 416,
 'seconds': 151.91692384600003,
 'mean_training_loss': 0.032193792276442625,
 'mean_gradient_norm': 1.1817757040978625,
 'clipped_updates': 89,
 'memory': {'label': 'epoch-2-complete',
  'allocated_gib': 64.96829175949097,
  'reserved_gib': 119.490234375,
  'peak_allocated_gib': 95.76671409606934,
  'peak_reserved_gib': 119.490234375},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 0,
 'transfer_panels_generated': []}

In [14]:
epoch_result = run_next_epoch()
epoch_result

Epoch 3/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 468, 'losses': {'seen_training_augmentation': 0.020512772724032402, 'canonical_original': 0.022006772458553314, 'unseen_augmentation': 0.021238353103399277}, 'memory': {'label': 'probe-step-468', 'allocated_gib': 64.99881649017334, 'reserved_gib': 119.490234375, 'peak_allocated_gib': 95.76548194885254, 'peak_reserved_gib': 119.490234375}}
Fixed probes: {'global_step': 520, 'losses': {'seen_training_augmentation': 0.017692623659968376, 'canonical_original': 0.018687019124627113, 'unseen_augmentation': 0.018255747854709625}, 'memory': {'label': 'probe-step-520', 'allocated_gib': 65.00262212753296, 'reserved_gib': 119.490234375, 'peak_allocated_gib': 95.7655200958252, 'peak_reserved_gib': 119.490234375}}
Fixed probes: {'global_step': 572, 'losses': {'seen_training_augmentation': 0.01616550050675869, 'canonical_original': 0.017220256850123405, 'unseen_augmentation': 0.01643306016921997}, 'memory': {'label': 'probe-step-572', 'allocated_gib': 64.99891519546509,

seen_training_augmentation @ step 624:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 624, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'seconds': 16.531977556000015, 'memory': {'label': 'seen_training_augmentation-generation-step-624', 'allocated_gib': 64.9680552482605, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76560592651367, 'peak_reserved_gib': 119.4921875}}


{'epoch': 3,
 'first_global_step': 417,
 'last_global_step': 624,
 'seconds': 152.17859540299992,
 'mean_training_loss': 0.0225078763581293,
 'mean_gradient_norm': 0.8570345285205314,
 'clipped_updates': 57,
 'memory': {'label': 'epoch-3-complete',
  'allocated_gib': 64.9680552482605,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76560592651367,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 0,
 'transfer_panels_generated': []}

In [15]:
epoch_result = run_next_epoch()
epoch_result

Epoch 4/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 676, 'losses': {'seen_training_augmentation': 0.01366855576634407, 'canonical_original': 0.014781396836042404, 'unseen_augmentation': 0.013973218388855457}, 'memory': {'label': 'probe-step-676', 'allocated_gib': 65.0013918876648, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76425361633301, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 728, 'losses': {'seen_training_augmentation': 0.013476761989295483, 'canonical_original': 0.015151061117649078, 'unseen_augmentation': 0.01426564808934927}, 'memory': {'label': 'probe-step-728', 'allocated_gib': 64.99783945083618, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76658630371094, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 780, 'losses': {'seen_training_augmentation': 0.014093305915594101, 'canonical_original': 0.014859907329082489, 'unseen_augmentation': 0.014734319411218166}, 'memory': {'label': 'probe-step-780', 'allocated_gib': 64.99940633773804, 'reserv

seen_training_augmentation @ step 832:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 832, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 7, 'correct_shape_pass2': 0, 'best_cell_accuracies': [None, None, None, None, None, None, None, None], 'seconds': 16.563729003999924, 'memory': {'label': 'seen_training_augmentation-generation-step-832', 'allocated_gib': 64.96829175949097, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76694679260254, 'peak_reserved_gib': 119.4921875}}


canonical_original @ step 832:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'canonical_original', 'view_name': 'train_original', 'global_step': 832, 'keys': ['3c9b0459_ID_0123456789_0,1,2,3,4', '0d3d703e_ID_0123456789_0,1,2,3,4', '8be77c9e_ID_0123456789_0,1,2,3', 'c9e6f938_ID_0123456789_0,1,2,3', '00576224_ID_0123456789_0,1,2', '0520fde7_ID_0123456789_0,1,2,3', '88a62173_ID_0123456789_0,1,2,3', 'd631b094_ID_0123456789_0,1,2,3,4'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 1, 'best_cell_accuracies': [None, None, None, None, 0.0833333358168602, None, None, None], 'seconds': 16.58089068700008, 'memory': {'label': 'canonical_original-generation-step-832', 'allocated_gib': 64.96829175949097, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76694679260254, 'peak_reserved_gib': 119.4921875}}


unseen_augmentation @ step 832:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'unseen_augmentation', 'view_name': 'train_unseen', 'global_step': 832, 'keys': ['3c9b0459_ID_5896310427_4,2,1,0,3', '0d3d703e_ID_4912687530_0,3,4,1,2', '8be77c9e_ID_9741652038_1,0,3,2', 'c9e6f938_ID_2675430198_3,2,1,0', '00576224_ID_0378942156_0,2,1', '0520fde7_ID_2830971546_0,2,1,3', '88a62173_ID_6017325489_2,3,0,1', 'd631b094_ID_7169453802_4,1,0,3,2'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 7, 'correct_shape_pass2': 1, 'best_cell_accuracies': [None, None, None, None, 0.0833333358168602, None, None, None], 'seconds': 16.56968637600005, 'memory': {'label': 'unseen_augmentation-generation-step-832', 'allocated_gib': 64.96829175949097, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76694679260254, 'peak_reserved_gib': 119.4921875}}


{'epoch': 4,
 'first_global_step': 625,
 'last_global_step': 832,
 'seconds': 151.993529609,
 'mean_training_loss': 0.017404858090090923,
 'mean_gradient_norm': 0.6012561325198755,
 'clipped_updates': 32,
 'memory': {'label': 'epoch-4-complete',
  'allocated_gib': 64.96829175949097,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76694679260254,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 0,
 'transfer_panels_generated': ['canonical_original', 'unseen_augmentation']}

In [16]:
epoch_result = run_next_epoch()
epoch_result

Epoch 5/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 884, 'losses': {'seen_training_augmentation': 0.011618038639426231, 'canonical_original': 0.012738511897623539, 'unseen_augmentation': 0.0120646758005023}, 'memory': {'label': 'probe-step-884', 'allocated_gib': 65.00013971328735, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76573371887207, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 936, 'losses': {'seen_training_augmentation': 0.012274454347789288, 'canonical_original': 0.013844740577042103, 'unseen_augmentation': 0.013217311352491379}, 'memory': {'label': 'probe-step-936', 'allocated_gib': 64.99841213226318, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76573371887207, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 988, 'losses': {'seen_training_augmentation': 0.014260705560445786, 'canonical_original': 0.015231150202453136, 'unseen_augmentation': 0.014758400619029999}, 'memory': {'label': 'probe-step-988', 'allocated_gib': 64.99935245513916, 'reser

seen_training_augmentation @ step 1040:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 1040, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 7, 'correct_shape_pass2': 2, 'best_cell_accuracies': [0.1111111119389534, 0.2222222238779068, None, None, None, None, None, None], 'seconds': 16.56224254499989, 'memory': {'label': 'seen_training_augmentation-generation-step-1040', 'allocated_gib': 64.96825361251831, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76573371887207, 'peak_reserved_gib': 119.4921875}}


{'epoch': 5,
 'first_global_step': 833,
 'last_global_step': 1040,
 'seconds': 152.0195142409999,
 'mean_training_loss': 0.016410258230350267,
 'mean_gradient_norm': 0.5725563759915531,
 'clipped_updates': 31,
 'memory': {'label': 'epoch-5-complete',
  'allocated_gib': 64.96825361251831,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76573371887207,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 2,
 'transfer_panels_generated': []}

In [17]:
epoch_result = run_next_epoch()
epoch_result

Epoch 6/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 1092, 'losses': {'seen_training_augmentation': 0.013460701331496239, 'canonical_original': 0.01433861069381237, 'unseen_augmentation': 0.013631259091198444}, 'memory': {'label': 'probe-step-1092', 'allocated_gib': 64.99973917007446, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.7651424407959, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1144, 'losses': {'seen_training_augmentation': 0.013944738544523716, 'canonical_original': 0.01515289954841137, 'unseen_augmentation': 0.014280376024544239}, 'memory': {'label': 'probe-step-1144', 'allocated_gib': 65.00205898284912, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76537895202637, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1196, 'losses': {'seen_training_augmentation': 0.011867450550198555, 'canonical_original': 0.012647249735891819, 'unseen_augmentation': 0.012323113158345222}, 'memory': {'label': 'probe-step-1196', 'allocated_gib': 65.00199222564697, '

seen_training_augmentation @ step 1248:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 1248, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 5, 'best_cell_accuracies': [0.0, 0.2222222238779068, 0.3888888955116272, None, 0.1944444477558136, 0.0, None, None], 'seconds': 16.57929176000016, 'memory': {'label': 'seen_training_augmentation-generation-step-1248', 'allocated_gib': 64.96898603439331, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76605796813965, 'peak_reserved_gib': 119.4921875}}


{'epoch': 6,
 'first_global_step': 1041,
 'last_global_step': 1248,
 'seconds': 152.10260963899987,
 'mean_training_loss': 0.019643401099109233,
 'mean_gradient_norm': 0.6046339273094558,
 'clipped_updates': 35,
 'memory': {'label': 'epoch-6-complete',
  'allocated_gib': 64.96898603439331,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76605796813965,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 5,
 'transfer_panels_generated': []}

In [18]:
epoch_result = run_next_epoch()
epoch_result

Epoch 7/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 1300, 'losses': {'seen_training_augmentation': 0.01414302084594965, 'canonical_original': 0.015180562622845173, 'unseen_augmentation': 0.014752756804227829}, 'memory': {'label': 'probe-step-1300', 'allocated_gib': 64.99779748916626, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76363182067871, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1352, 'losses': {'seen_training_augmentation': 0.013242841698229313, 'canonical_original': 0.013787566684186459, 'unseen_augmentation': 0.013943608850240707}, 'memory': {'label': 'probe-step-1352', 'allocated_gib': 64.9986047744751, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.7654800415039, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1404, 'losses': {'seen_training_augmentation': 0.016616130247712135, 'canonical_original': 0.01762210763990879, 'unseen_augmentation': 0.017009178176522255}, 'memory': {'label': 'probe-step-1404', 'allocated_gib': 65.00382375717163, 'r

seen_training_augmentation @ step 1456:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 1456, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 5, 'best_cell_accuracies': [0.1111111119389534, 0.0, None, 0.5, None, 0.0, 0.0, None], 'seconds': 16.571142492000035, 'memory': {'label': 'seen_training_augmentation-generation-step-1456', 'allocated_gib': 64.96922254562378, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.7654800415039, 'peak_reserved_gib': 119.4921875}}


{'epoch': 7,
 'first_global_step': 1249,
 'last_global_step': 1456,
 'seconds': 151.7715357259999,
 'mean_training_loss': 0.017693093017442152,
 'mean_gradient_norm': 0.46174601329347265,
 'clipped_updates': 17,
 'memory': {'label': 'epoch-7-complete',
  'allocated_gib': 64.96922254562378,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.7654800415039,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 5,
 'transfer_panels_generated': []}

In [19]:
epoch_result = run_next_epoch()
epoch_result

Epoch 8/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 1508, 'losses': {'seen_training_augmentation': 0.011622047983109951, 'canonical_original': 0.012703590095043182, 'unseen_augmentation': 0.012250682339072227}, 'memory': {'label': 'probe-step-1508', 'allocated_gib': 65.00021600723267, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76529884338379, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1560, 'losses': {'seen_training_augmentation': 0.012353473342955112, 'canonical_original': 0.013499145396053791, 'unseen_augmentation': 0.012799547053873539}, 'memory': {'label': 'probe-step-1560', 'allocated_gib': 65.00166988372803, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76529884338379, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1612, 'losses': {'seen_training_augmentation': 0.011999274604022503, 'canonical_original': 0.01383399497717619, 'unseen_augmentation': 0.011883782222867012}, 'memory': {'label': 'probe-step-1612', 'allocated_gib': 65.00425815582275,

seen_training_augmentation @ step 1664:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 1664, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 6, 'best_cell_accuracies': [0.0, None, 0.3888888955116272, None, 0.25, 0.0, 0.0, 0.0], 'seconds': 16.628162907999922, 'memory': {'label': 'seen_training_augmentation-generation-step-1664', 'allocated_gib': 64.96788740158081, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76529884338379, 'peak_reserved_gib': 119.4921875}}


canonical_original @ step 1664:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'canonical_original', 'view_name': 'train_original', 'global_step': 1664, 'keys': ['3c9b0459_ID_0123456789_0,1,2,3,4', '0d3d703e_ID_0123456789_0,1,2,3,4', '8be77c9e_ID_0123456789_0,1,2,3', 'c9e6f938_ID_0123456789_0,1,2,3', '00576224_ID_0123456789_0,1,2', '0520fde7_ID_0123456789_0,1,2,3', '88a62173_ID_0123456789_0,1,2,3', 'd631b094_ID_0123456789_0,1,2,3,4'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 5, 'best_cell_accuracies': [0.2222222238779068, None, 0.1111111119389534, None, 0.02777777798473835, 0.3333333432674408, 0.0, None], 'seconds': 16.621745965999935, 'memory': {'label': 'canonical_original-generation-step-1664', 'allocated_gib': 64.96788740158081, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76529884338379, 'peak_reserved_gib': 119.4921875}}


unseen_augmentation @ step 1664:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'unseen_augmentation', 'view_name': 'train_unseen', 'global_step': 1664, 'keys': ['3c9b0459_ID_5896310427_4,2,1,0,3', '0d3d703e_ID_4912687530_0,3,4,1,2', '8be77c9e_ID_9741652038_1,0,3,2', 'c9e6f938_ID_2675430198_3,2,1,0', '00576224_ID_0378942156_0,2,1', '0520fde7_ID_2830971546_0,2,1,3', '88a62173_ID_6017325489_2,3,0,1', 'd631b094_ID_7169453802_4,1,0,3,2'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 5, 'best_cell_accuracies': [0.0, None, 0.0, None, 0.0833333358168602, 0.3333333432674408, 0.0, None], 'seconds': 16.604906880000044, 'memory': {'label': 'unseen_augmentation-generation-step-1664', 'allocated_gib': 64.96788740158081, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76529884338379, 'peak_reserved_gib': 119.4921875}}


{'epoch': 8,
 'first_global_step': 1457,
 'last_global_step': 1664,
 'seconds': 152.57014947800008,
 'mean_training_loss': 0.01639576827381881,
 'mean_gradient_norm': 0.5409910924589405,
 'clipped_updates': 28,
 'memory': {'label': 'epoch-8-complete',
  'allocated_gib': 64.96788740158081,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76529884338379,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 6,
 'transfer_panels_generated': ['canonical_original', 'unseen_augmentation']}

In [20]:
epoch_result = run_next_epoch()
epoch_result

Epoch 9/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 1716, 'losses': {'seen_training_augmentation': 0.014129948802292347, 'canonical_original': 0.014804895967245102, 'unseen_augmentation': 0.014358234591782093}, 'memory': {'label': 'probe-step-1716', 'allocated_gib': 64.99645614624023, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76628112792969, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1768, 'losses': {'seen_training_augmentation': 0.00982809066772461, 'canonical_original': 0.01174247357994318, 'unseen_augmentation': 0.0100522730499506}, 'memory': {'label': 'probe-step-1768', 'allocated_gib': 65.0039496421814, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76714515686035, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1820, 'losses': {'seen_training_augmentation': 0.012956473976373672, 'canonical_original': 0.014083292335271835, 'unseen_augmentation': 0.013403362594544888}, 'memory': {'label': 'probe-step-1820', 'allocated_gib': 64.99966669082642, 're

seen_training_augmentation @ step 1872:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 1872, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 5, 'best_cell_accuracies': [0.0, 0.0, 0.0, None, None, 0.0, 0.0, None], 'seconds': 16.648914792999904, 'memory': {'label': 'seen_training_augmentation-generation-step-1872', 'allocated_gib': 64.96830701828003, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76714515686035, 'peak_reserved_gib': 119.4921875}}


{'epoch': 9,
 'first_global_step': 1665,
 'last_global_step': 1872,
 'seconds': 154.45094283699996,
 'mean_training_loss': 0.013839753349132549,
 'mean_gradient_norm': 0.3908234304175354,
 'clipped_updates': 20,
 'memory': {'label': 'epoch-9-complete',
  'allocated_gib': 64.96830701828003,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76714515686035,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 5,
 'transfer_panels_generated': []}

In [21]:
epoch_result = run_next_epoch()
epoch_result

Epoch 10/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 1924, 'losses': {'seen_training_augmentation': 0.010384779423475266, 'canonical_original': 0.011155487038195133, 'unseen_augmentation': 0.011315727606415749}, 'memory': {'label': 'probe-step-1924', 'allocated_gib': 65.00606679916382, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76548194885254, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 1976, 'losses': {'seen_training_augmentation': 0.010571984574198723, 'canonical_original': 0.011856789700686932, 'unseen_augmentation': 0.011355879716575146}, 'memory': {'label': 'probe-step-1976', 'allocated_gib': 64.99907779693604, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76548194885254, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 2028, 'losses': {'seen_training_augmentation': 0.011935780756175518, 'canonical_original': 0.013550828211009502, 'unseen_augmentation': 0.012638679705560207}, 'memory': {'label': 'probe-step-2028', 'allocated_gib': 64.99922466278076

seen_training_augmentation @ step 2080:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 2080, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 6, 'best_cell_accuracies': [0.3333333432674408, 0.0, 0.1666666716337204, 0.0, 0.1666666716337204, 0.1111111119389534, None, None], 'seconds': 16.574167665000004, 'memory': {'label': 'seen_training_augmentation-generation-step-2080', 'allocated_gib': 64.96788740158081, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76548194885254, 'peak_reserved_gib': 119.4921875}}


{'epoch': 10,
 'first_global_step': 1873,
 'last_global_step': 2080,
 'seconds': 151.89602340800002,
 'mean_training_loss': 0.016837612938476153,
 'mean_gradient_norm': 0.46164757378685933,
 'clipped_updates': 21,
 'memory': {'label': 'epoch-10-complete',
  'allocated_gib': 64.96788740158081,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76548194885254,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 6,
 'transfer_panels_generated': []}

In [22]:
epoch_result = run_next_epoch()
epoch_result

Epoch 11/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 2132, 'losses': {'seen_training_augmentation': 0.01007691491395235, 'canonical_original': 0.011450990103185177, 'unseen_augmentation': 0.010945449583232403}, 'memory': {'label': 'probe-step-2132', 'allocated_gib': 65.0007004737854, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76383399963379, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 2184, 'losses': {'seen_training_augmentation': 0.010192644782364368, 'canonical_original': 0.011403302662074566, 'unseen_augmentation': 0.010651242919266224}, 'memory': {'label': 'probe-step-2184', 'allocated_gib': 65.0003457069397, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76407051086426, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 2236, 'losses': {'seen_training_augmentation': 0.01086387038230896, 'canonical_original': 0.012204466387629509, 'unseen_augmentation': 0.011398669332265854}, 'memory': {'label': 'probe-step-2236', 'allocated_gib': 64.99887132644653, 'r

seen_training_augmentation @ step 2288:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 2288, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 7, 'best_cell_accuracies': [0.3333333432674408, 0.0, 0.3888888955116272, 0.0555555559694767, 0.1666666716337204, 0.1111111119389534, 0.0, None], 'seconds': 16.58342341000025, 'memory': {'label': 'seen_training_augmentation-generation-step-2288', 'allocated_gib': 64.96922254562378, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76694679260254, 'peak_reserved_gib': 119.4921875}}


{'epoch': 11,
 'first_global_step': 2081,
 'last_global_step': 2288,
 'seconds': 152.01960235299975,
 'mean_training_loss': 0.01417351966105903,
 'mean_gradient_norm': 0.42261680704541504,
 'clipped_updates': 20,
 'memory': {'label': 'epoch-11-complete',
  'allocated_gib': 64.96922254562378,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76694679260254,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 7,
 'transfer_panels_generated': []}

In [23]:
epoch_result = run_next_epoch()
epoch_result

Epoch 12/12:   0%|          | 0/208 [00:00<?, ?update/s]

Fixed probes: {'global_step': 2340, 'losses': {'seen_training_augmentation': 0.009922578930854797, 'canonical_original': 0.01097674947232008, 'unseen_augmentation': 0.010374212637543678}, 'memory': {'label': 'probe-step-2340', 'allocated_gib': 65.00151491165161, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76527214050293, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 2392, 'losses': {'seen_training_augmentation': 0.010470805689692497, 'canonical_original': 0.01152694970369339, 'unseen_augmentation': 0.010740302503108978}, 'memory': {'label': 'probe-step-2392', 'allocated_gib': 65.00215291976929, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76615715026855, 'peak_reserved_gib': 119.4921875}}
Fixed probes: {'global_step': 2444, 'losses': {'seen_training_augmentation': 0.009325082413852215, 'canonical_original': 0.010842233896255493, 'unseen_augmentation': 0.00934786070138216}, 'memory': {'label': 'probe-step-2444', 'allocated_gib': 65.0021071434021, 'r

seen_training_augmentation @ step 2496:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'seen_training_augmentation', 'view_name': 'train', 'global_step': 2496, 'keys': ['3c9b0459_ID_1290374658_4,1,2,3,0', '0d3d703e_ID_7458913620_3,0,4,2,1', '8be77c9e_ID_6921504738_1,2,3,0', 'c9e6f938_ID_7809641352_0,1,3,2', '00576224_ID_6095374821_1,0,2', '0520fde7_ID_4960851372_2,0,1,3', '88a62173_ID_1984750623_2,0,1,3', 'd631b094_ID_0487193526_2,4,0,1,3'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 7, 'best_cell_accuracies': [0.0, 0.0, 0.5555555820465088, 0.1111111119389534, 0.1666666716337204, 0.0, 0.0, None], 'seconds': 16.588374178999857, 'memory': {'label': 'seen_training_augmentation-generation-step-2496', 'allocated_gib': 64.96830701828003, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76615715026855, 'peak_reserved_gib': 119.4921875}}


canonical_original @ step 2496:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'canonical_original', 'view_name': 'train_original', 'global_step': 2496, 'keys': ['3c9b0459_ID_0123456789_0,1,2,3,4', '0d3d703e_ID_0123456789_0,1,2,3,4', '8be77c9e_ID_0123456789_0,1,2,3', 'c9e6f938_ID_0123456789_0,1,2,3', '00576224_ID_0123456789_0,1,2', '0520fde7_ID_0123456789_0,1,2,3', '88a62173_ID_0123456789_0,1,2,3', 'd631b094_ID_0123456789_0,1,2,3,4'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 7, 'best_cell_accuracies': [0.3333333432674408, 0.3333333432674408, 0.0, 0.1111111119389534, 0.2222222238779068, 0.2222222238779068, 0.0, None], 'seconds': 16.61511475999987, 'memory': {'label': 'canonical_original-generation-step-2496', 'allocated_gib': 64.96830701828003, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76615715026855, 'peak_reserved_gib': 119.4921875}}


unseen_augmentation @ step 2496:   0%|          | 0/2 [00:00<?, ?seed/s]

{'panel': 'unseen_augmentation', 'view_name': 'train_unseen', 'global_step': 2496, 'keys': ['3c9b0459_ID_5896310427_4,2,1,0,3', '0d3d703e_ID_4912687530_0,3,4,1,2', '8be77c9e_ID_9741652038_1,0,3,2', 'c9e6f938_ID_2675430198_3,2,1,0', '00576224_ID_0378942156_0,2,1', '0520fde7_ID_2830971546_0,2,1,3', '88a62173_ID_6017325489_2,3,0,1', 'd631b094_ID_7169453802_4,1,0,3,2'], 'exact_grid_pass1': 0, 'exact_grid_pass2': 0, 'valid_grid_pass2': 8, 'correct_shape_pass2': 7, 'best_cell_accuracies': [0.1111111119389534, 0.6666666865348816, 0.0, 0.0, 0.1388888955116272, 0.5555555820465088, 0.0, None], 'seconds': 16.603719397000077, 'memory': {'label': 'unseen_augmentation-generation-step-2496', 'allocated_gib': 64.96830701828003, 'reserved_gib': 119.4921875, 'peak_allocated_gib': 95.76615715026855, 'peak_reserved_gib': 119.4921875}}


{'epoch': 12,
 'first_global_step': 2289,
 'last_global_step': 2496,
 'seconds': 152.04129761100012,
 'mean_training_loss': 0.014310097616264382,
 'mean_gradient_norm': 0.39383015803133065,
 'clipped_updates': 14,
 'memory': {'label': 'epoch-12-complete',
  'allocated_gib': 64.96830701828003,
  'reserved_gib': 119.4921875,
  'peak_allocated_gib': 95.76615715026855,
  'peak_reserved_gib': 119.4921875},
 'seen_exact_grid_pass2': 0,
 'seen_correct_shape_pass2': 7,
 'transfer_panels_generated': ['canonical_original', 'unseen_augmentation']}

In [24]:
summary = evidence_summary()
print('Status:', REPORT['status'])
print('Completed epochs:', COMPLETED_EPOCHS)
print('Completed optimizer steps:', GLOBAL_STEP)
print('Latest seen exact-grid pass@2:', summary['latest_seen_exact_grid_pass2'], '/ 8')
print('Latest seen correct-shape pass@2:', summary['latest_seen_correct_shape_pass2'], '/ 8')
print('Report:', REPORT_PATH)
summary

Status: TRAINING_LIMIT_REACHED
Completed epochs: 12
Completed optimizer steps: 2496
Latest seen exact-grid pass@2: 0 / 8
Latest seen correct-shape pass@2: 7 / 8
Report: /mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T150620231361Z/report.json


{'primary_panel': 'seen_training_augmentation',
 'train_probe_loss_ratio': 0.06810418363268742,
 'optimization_signal': True,
 'positive_overfitting_evidence': False,
 'strong_overfitting_evidence': False,
 'complete_overfitting': False,
 'baseline_conditional_cell_accuracy': None,
 'final_conditional_cell_accuracy': 0.11904762365988322,
 'latest_seen_exact_grid_pass2': 0,
 'latest_seen_correct_shape_pass2': 7}

## 10. Explicit durable checkpoint finalization

Run this cell once when you intentionally stop training. It may take
several minutes because it saves and hashes the FP32 transformer, commits
and reloads the Volume for staging, final rename, and the report, then
verifies the durable checkpoint and report. Do not stop the kernel while
this cell is running.

If it raises, the automatic shutdown cell remains blocked. The in-memory
model is still available for diagnosis or another explicitly chosen save
attempt.


In [25]:
saved_checkpoint = save_current_checkpoint()
saved_checkpoint


Saving transformer to local ephemeral storage: /tmp/flux-transformer-step-2496-20260825T150620231361Z
Local checkpoint verified: 11 files in 27.5s
Copying verified checkpoint to Volume staging: /mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T150620231361Z/.flux-transformer-step-2496.partial-c9c70c21fb254c55a9f6fcf4544df352


RuntimeError: commit() can only be called on a mounted volume inside a container

In [26]:
from pathlib import Path
import modal

checkpoint_name = f"flux-transformer-step-{GLOBAL_STEP:04d}"
local_dir = Path("/tmp") / f"{checkpoint_name}-{RUN_ID}"
final_dir = REPORT_DIR / checkpoint_name
staging_dirs = sorted(REPORT_DIR.glob(f".{checkpoint_name}.partial-*"))

def show_directory(label, directory):
    print(f"\n{label}: {directory}")
    print("exists:", directory.exists())
    if directory.exists():
        for file in sorted(directory.rglob("*")):
            if file.is_file():
                print(
                    f"  {file.relative_to(directory)} "
                    f"({file.stat().st_size / 1024**3:.3f} GiB)"
                )

print("RUN_ID:", RUN_ID)
print("GLOBAL_STEP:", GLOBAL_STEP)
print("COMPLETED_EPOCHS:", COMPLETED_EPOCHS)
print("REPORT status:", REPORT.get("status"))
print("REPORT failures:", REPORT.get("failures"))
print("staging directories:", staging_dirs)

show_directory("Verified local checkpoint", local_dir)
for staging_dir in staging_dirs:
    show_directory("Mounted staging checkpoint", staging_dir)
show_directory("Final checkpoint", final_dir)

remote_volume = modal.Volume.from_name(
    VOLUME_NAME,
    environment_name="diffusion-arc",
)
print("\nRemote volume info:")
print(remote_volume.info())

remote_run_path = (
    f"results/single-gpu-overfitting-pilot/{RUN_ID}"
)
print("\nDurably visible remote entries:")
try:
    for entry in remote_volume.iterdir(remote_run_path, recursive=True):
        print(entry)
except Exception as error:
    print(type(error).__name__, error)

RUN_ID: 20260825T150620231361Z
GLOBAL_STEP: 2496
COMPLETED_EPOCHS: 12
REPORT status: FAIL
REPORT failures: [{'stage': 'checkpoint-finalization', 'type': 'RuntimeError', 'message': 'commit() can only be called on a mounted volume inside a container', 'recorded_at_utc': '2026-08-25T15:48:23.748361+00:00'}]
staging directories: [PosixPath('/mnt/diffusion-arc-models/results/single-gpu-overfitting-pilot/20260825T150620231361Z/.flux-transformer-step-2496.partial-c9c70c21fb254c55a9f6fcf4544df352')]

Verified local checkpoint: /tmp/flux-transformer-step-2496-20260825T150620231361Z
exists: True
  config.json (0.000 GiB)
  diffusion_pytorch_model-00001-of-00009.safetensors (1.850 GiB)
  diffusion_pytorch_model-00002-of-00009.safetensors (1.828 GiB)
  diffusion_pytorch_model-00003-of-00009.safetensors (1.547 GiB)
  diffusion_pytorch_model-00004-of-00009.safetensors (1.828 GiB)
  diffusion_pytorch_model-00005-of-00009.safetensors (1.828 GiB)
  diffusion_pytorch_model-00006-of-00009.safetensors (1.

## 11. Optional automatic shutdown

This cell schedules shutdown only after the reloaded report records a
completely verified checkpoint. Set `DIFFUSION_ARC_AUTO_SHUTDOWN=0` before
starting the notebook to keep the kernel alive.


In [ ]:
durable_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
saved = durable_report.get('saved_checkpoint')
if durable_report.get('status') != 'COMPLETE':
    raise RuntimeError('Refusing shutdown: durable report is not COMPLETE')
if not isinstance(saved, dict) or not saved.get('volume_commit_verified'):
    raise RuntimeError('Refusing shutdown: durable checkpoint is not verified')
if not AUTO_SHUTDOWN:
    print('Automatic shutdown disabled; checkpoint is durable:', saved['path'])
else:
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    if not modal_task_id:
        raise RuntimeError('MODAL_TASK_ID is unavailable; stop the kernel manually')
    shutdown_code = (
        'import subprocess, sys, time; '
        'time.sleep(10); '
        "subprocess.run(['modal', 'container', 'stop', '--yes', sys.argv[1]], check=False)"
    )
    subprocess.Popen(
        [sys.executable, '-c', shutdown_code, modal_task_id],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    print('Checkpoint verified. Modal container shutdown scheduled in 10 seconds.')
